In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import warnings

import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.config import set_seed
from dataeval.data import Filter, Limit, Shuffle, View

set_seed(0)  # For reproducibility

In [3]:
# A 200-image sample of SeaDrone stands in for your dataset. This guide does not use the binned factors, so skip the
# warnings about binning them.
warnings.filterwarnings("ignore", message=".*(was dropped|binned automatically).*")
dataset = View(SeaDrone(root="./data", image_set="val", download=True), [Shuffle(seed=0), Limit(200)])

In [4]:
print("all images:", Metadata(dataset).level_counts)

high = View(dataset, Filter("drone = 'mavic' AND altitude > 60"))
print("mavic above 60 m:", Metadata(high).level_counts)

all images: {'unit': 200, 'instance': 1305}


mavic above 60 m: {'unit': 15, 'instance': 151}


In [5]:
high_polars = View(dataset, Filter((pl.col("drone") == "mavic") & (pl.col("altitude") > 60)))

print("mavic above 60 m:", Metadata(high_polars).level_counts)

mavic above 60 m: {'unit': 15, 'instance': 151}


In [6]:
with_telemetry = View(dataset, Filter("altitude >= 0"))

print("images with telemetry:", len(with_telemetry), "of", len(dataset))

images with telemetry: 142 of 200


In [7]:
for drop in ("both", "items", "detections"):
    kept = View(dataset, Filter("object_size > 10000", drop=drop))
    print(f"drop={drop!r:13}", Metadata(kept).level_counts)

drop='both'        {'unit': 117, 'instance': 237}


drop='items'       {'unit': 117, 'instance': 819}


drop='detections'  {'unit': 200, 'instance': 237}


In [8]:
swimmers_high = View(dataset, Filter("altitude > 60", classes=[1]))

print("swimmers above 60 m:", Metadata(swimmers_high).level_counts)

swimmers above 60 m: {'unit': 16, 'instance': 107}


In [9]:
# TEST ASSERTION CELL ###
assert Metadata(dataset).level_counts == {"unit": 200, "instance": 1305}
assert Metadata(high).level_counts == {"unit": 15, "instance": 151}
assert high_polars.resolve_indices() == high.resolve_indices()
assert len(with_telemetry) == 142
sizes = {
    drop: Metadata(View(dataset, Filter("object_size > 10000", drop=drop))).level_counts
    for drop in ("both", "items", "detections")
}
assert sizes["both"] == {"unit": 117, "instance": 237}
assert sizes["items"] == {"unit": 117, "instance": 819}
assert sizes["detections"] == {"unit": 200, "instance": 237}
assert Metadata(swimmers_high).level_counts == {"unit": 16, "instance": 107}